In [ ]:
# ==========================================
# MODULO 2: REGISTRO DE USUARIO
# ==========================================

def registrar_usuario():

    print("\nREGISTRO DE NUEVO USUARIO")
    print("-------------------------")

    # Consultamos los usuarios existentes

    usuarios = []

    if os.path.exists(archivo_usuarios):

        with open(archivo_usuarios, "r", encoding="utf-8") as archivo:

            lector = csv.DictReader(archivo)

            usuarios = list(lector)

    # Solicitamos los datos personales

    nombre = input("Nombre completo: ")

    edad = int(input("Edad: "))

    carrera = input("Carrera: ")

    semestre = int(input("Semestre: "))

    recibe_beca = input("¿Recibe beca? (Si/No): ")

    # Creamos el nombre de usuario

    usuario = input("Crea tu nombre de usuario: ").strip().lower()

    # Verificamos que no exista

    for persona in usuarios:

        if persona["usuario"] == usuario:

            print("Este nombre de usuario ya existe")

            return

    # Solicitamos la contraseña

    clave = getpass("Crea tu contraseña: ")

    confirmar_clave = getpass("Confirma tu contraseña: ")

    if clave != confirmar_clave:

        print("Las contraseñas no coinciden")

        return

    # Protegemos la contraseña mediante un hash

    sal = secrets.token_hex(16)

    clave_hash = hashlib.pbkdf2_hmac(
        "sha256",
        clave.encode(),
        bytes.fromhex(sal),
        100000
    ).hex()

    # Creamos el registro del usuario

    nuevo_usuario = {
        "id_usuario": len(usuarios) + 1,
        "nombre": nombre,
        "edad": edad,
        "carrera": carrera,
        "semestre": semestre,
        "recibe_beca": recibe_beca,
        "usuario": usuario,
        "sal": sal,
        "clave_hash": clave_hash
    }

    # Guardamos la información en CSV

    campos = list(nuevo_usuario.keys())

    existe = os.path.exists(archivo_usuarios)

    with open(archivo_usuarios, "a", newline="", encoding="utf-8") as archivo:

        escritor = csv.DictWriter(archivo, fieldnames=campos)

        if not existe or os.path.getsize(archivo_usuarios) == 0:

            escritor.writeheader()

        escritor.writerow(nuevo_usuario)

    print("\n¡REGISTRO COMPLETADO!")

    print("Bienvenido a MoniApps")

In [ ]:
# ==========================================
# MODULO 8: CONSULTA DEL BALANCE FINANCIERO
# ==========================================

# Consultamos el mes y año actual

mes_actual = date.today().strftime("%Y-%m")


def consultar_balance(usuario_activo):

    # Verificamos que exista una sesion activa

    if usuario_activo is None:
        print("Primero debes iniciar sesion.")
        return

    # Leemos los movimientos del estudiante, si existen

    movimientos = []

    if os.path.exists(archivo_movimientos):

        with open(archivo_movimientos, "r", encoding="utf-8") as archivo:

            lector = csv.DictReader(archivo)

            movimientos = [
                fila for fila in lector
                if fila["id_usuario"] == str(usuario_activo["id_usuario"])
                and fila["fecha"].startswith(mes_actual)
            ]

    # Consultamos el saldo inicial del estudiante

    saldo_inicial = 0

    if os.path.exists(archivo_saldo_inicial):

        with open(archivo_saldo_inicial, "r", encoding="utf-8") as archivo:

            lector = csv.DictReader(archivo)

            for fila in lector:

                if fila["id_usuario"] == str(usuario_activo["id_usuario"]):

                    saldo_inicial = float(fila["saldo_inicial"])
                    break

    # Calculamos los ingresos y gastos

    total_ingresos = 0
    total_gastos = 0

    gastos_por_categoria = {}

    for movimiento in movimientos:

        valor = float(movimiento["valor"])

        if movimiento["tipo"] == "Ingreso":

            total_ingresos += valor

        elif movimiento["tipo"] == "Gasto":

            total_gastos += valor

            categoria = movimiento["categoria"]

            gastos_por_categoria[categoria] = (
                gastos_por_categoria.get(categoria, 0) + valor
            )

    # Calculamos el dinero disponible incluyendo el saldo inicial

    balance = saldo_inicial + total_ingresos - total_gastos

    # Mostramos el balance financiero

    print("\nBALANCE FINANCIERO")
    print("-----------------------------")

    print("Estudiante:", usuario_activo["nombre"])
    print("Periodo:", mes_actual)

    print("\nRESUMEN DEL MES")
    print("-----------------------------")

    print("Saldo inicial: $", format(saldo_inicial, ",.0f"))
    print("Ingresos del mes: $", format(total_ingresos, ",.0f"))
    print("Gastos del mes: $", format(total_gastos, ",.0f"))
    print("Dinero disponible: $", format(balance, ",.0f"))

    # Consultamos el presupuesto del estudiante

    if os.path.exists(archivo_presupuestos):

        with open(archivo_presupuestos, "r", encoding="utf-8") as archivo:

            lector = csv.DictReader(archivo)

            presupuestos = [
                fila for fila in lector
                if fila["id_usuario"] == str(usuario_activo["id_usuario"])
            ]

        if presupuestos:

            total_presupuestado = sum(
                float(fila["valor_categoria"])
                for fila in presupuestos
            )

            print("\nCONTROL DEL PRESUPUESTO")
            print("-----------------------------")

            print(
                "Presupuesto asignado: $",
                format(total_presupuestado, ",.0f")
            )

            print(
                "Gastos registrados: $",
                format(total_gastos, ",.0f")
            )

            print(
                "Presupuesto restante: $",
                format(total_presupuestado - total_gastos, ",.0f")
            )

            print("\nDETALLE POR CATEGORIA")
            print("-----------------------------")

            for fila in presupuestos:

                categoria = fila["categoria"]

                presupuesto = float(fila["valor_categoria"])

                gastado = gastos_por_categoria.get(categoria, 0)

                restante = presupuesto - gastado

                print("\nCategoria:", categoria)
                print("Presupuesto: $", format(presupuesto, ",.0f"))
                print("Gastado: $", format(gastado, ",.0f"))
                print("Disponible: $", format(restante, ",.0f"))

    else:

        print("\nTodavia no tienes presupuesto configurado.")
